# Score your submission offline — and know how much to trust it

Kaggle notebooks have no internet, so you cannot `git clone` or `pip install` the
organisers' scoring package. This attaches it as a dataset and scores your
`submission.csv` against the training ground truth with the **official patched
metric**.

But the useful part is the second half. A local score on this competition's
public labels is **much less certain than it looks**, and this notebook tells you
by how much rather than quietly overstating it.

## The one number worth taking away

The four movies on the public leaderboard are also in `train/`, so you *can*
score them locally. The catch is coverage: they carry roughly **2,100 annotated
edges between them, against 25k–70k cells per movie**. The metric ignores any
edge whose endpoints match no annotated node, so a local score sees on the order
of **2% of your predictions**.

With `D ≈ 2,100` evaluable edges and `J ≈ 0.9`, sampling error alone is

```
SE ≈ sqrt(J(1-J)/D) ≈ 0.006      95% CI ≈ +-0.013
```

**Differences below roughly ±0.013 are not measurable with these labels.** If you
are choosing between two variants that differ by 0.002 locally, you are reading
noise. This notebook computes that bound from *your* numbers and says so plainly.

## What it does

1. loads the official **patched** metric (and asserts it really is the patched one)
2. scores your submission per movie and in aggregate
3. separates the two parts of the score - one is exact, one is sampled
4. puts a **bootstrap confidence interval** on the sampled part
5. states the resolution limit, and where your uncertainty comes from

Scoring code: [royerlab/kaggle-cell-tracking-competition](https://github.com/royerlab/kaggle-cell-tracking-competition)
(BSD-3), pinned at `075fc5f`. All credit to the organisers.


## 0. Offline dependencies

`tracksdata`, `geff` and `zarr` are not preinstalled. We install them from the
wheels in the public support pack.

**A trap worth knowing:** that same pack vendors its own copy of the scoring code
as `biohub_tracking`, and that copy **predates the 2026-07-17 division patch**. If
you already have the pack attached and do `from biohub_tracking.metrics import
evaluate`, your division numbers use the old, exploitable rule. We use the pack
only for wheels and import the metric from the pinned patched copy — cell 2
asserts we got the right one.

In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    # Polars wheel in the support package uses the 32-bit runtime package.
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        # Fallback for Kaggle's shorter mounted dataset path.
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    
    # These are packages supplied by the attached dataset.
    # Deliberately exclude:
    #   numpy, scipy, torch, pandas, dask, xarray, numba, llvmlite
    #
    # Kaggle already supplies compatible versions of those packages.
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    # Import name differs from pip package name in several cases.
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    # Verify the complete import chain needed by biohub_tracking.io.
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

In [ ]:
import sys, os, glob, json, csv, math
import numpy as np

cands = []
for pat in ("/kaggle/input/*/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/**/tracking_cellmot/metrics.py"):
    cands += glob.glob(pat, recursive=True)
assert cands, "official scorer dataset not attached"
SRC  = os.path.dirname(os.path.dirname(cands[0]))
REPO = os.path.dirname(SRC)
sys.path.insert(0, SRC)

import tracksdata as td
import polars as pl
from geff import GeffMetadata
from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import evaluate, node_recall, per_sample_metrics

import inspect, tracking_cellmot.division_metrics as dm
assert "_weakly_connected_components" not in inspect.getsource(dm), \
    "PRE-PATCH metric loaded - division scoring would use the exploitable rule"
print("official PATCHED metric loaded from", SRC)


## 1. Point this at your submission

Set `SUBMISSION_PATH` to your own file — attach the notebook that produced it, or
upload it as a dataset. If nothing is found, the notebook builds a demo
submission from the ground truth so it still runs end to end.

In [ ]:
COMP = "/kaggle/input/competitions/biohub-cell-tracking-during-development"
if not os.path.exists(COMP):
    COMP = glob.glob("/kaggle/input/*biohub-cell-tracking*")[0]
TRAIN = f"{COMP}/train"

# The four movies the public leaderboard scores. They are also in train/, which
# is the only reason local scoring is possible at all.
LB_MOVIES = ["44b6_0113de3b", "44b6_0b24845f", "6bba_05b6850b", "6bba_05db0fb1"]

SUBMISSION_PATH = None                      # <-- set yours here
if SUBMISSION_PATH is None:
    found = [p for p in glob.glob("/kaggle/input/**/submission.csv", recursive=True)]
    SUBMISSION_PATH = found[0] if found else None
print("submission:", SUBMISSION_PATH or "none found - will build a demo from GT")

available = [m for m in LB_MOVIES
             if os.path.exists(f"{TRAIN}/{m}.zarr") and os.path.exists(f"{TRAIN}/{m}.geff")]
print(f"LB movies with ground truth available: {len(available)}/{len(LB_MOVIES)}")
assert available, "none of the LB movies found in train/"


In [ ]:
# Load ground truth once per movie.
truth = {}
for m in available:
    ds = open_dataset(f"{TRAIN}/{m}.zarr", normalize=False, load_image=False, require_tracks=True)
    meta = GeffMetadata.read(f"{TRAIN}/{m}.geff")
    truth[m] = {"graph": ds.tracks, "scale": ds.scale,
                "n_total": float(meta.extra["estimated_number_of_nodes"])}
    print(f"{m:<18} GT edges {ds.tracks.num_edges():>5,}   "
          f"N_true {truth[m]['n_total']:>8,.0f}   "
          f"labelled {ds.tracks.num_nodes()/truth[m]['n_total']:>6.2%}")

def build_graph(rows_nodes, rows_edges):
    g = td.graph.InMemoryGraph()
    for k in ("z", "y", "x"):
        g.add_node_attr_key(k, pl.Float64, -999999.0)
    ids = [int(n["node_id"]) for n in rows_nodes]
    new = g.bulk_add_nodes([{"t": int(n["t"]), "z": float(n["z"]),
                             "y": float(n["y"]), "x": float(n["x"])} for n in rows_nodes])
    remap = dict(zip(ids, new))
    keep = [(s, t) for s, t in rows_edges if s in remap and t in remap]
    if keep:
        g.bulk_add_edges([{"source_id": remap[s], "target_id": remap[t]} for s, t in keep])
    return g

def load_submission(path):
    """Returns {dataset: (node_rows, edge_pairs)}. node_id is DATASET-LOCAL."""
    per = {}
    with open(path) as fh:
        for r in csv.DictReader(fh):
            d = r["dataset"]
            per.setdefault(d, ([], []))
            if r["row_type"] == "node":
                per[d][0].append({"node_id": int(r["node_id"]), "t": int(r["t"]),
                                  "z": float(r["z"]), "y": float(r["y"]), "x": float(r["x"])})
            else:
                per[d][1].append((int(r["source_id"]), int(r["target_id"])))
    return per

if SUBMISSION_PATH:
    pred = load_submission(SUBMISSION_PATH)
else:
    pred = {}
    for m in available:
        g = truth[m]["graph"]
        nrows = list(g.node_attrs().iter_rows(named=True))
        erows = list(g.edge_attrs().iter_rows(named=True))
        pred[m] = ([{ "node_id": int(r["node_id"]), "t": int(r["t"]),
                      "z": float(r["z"]), "y": float(r["y"]), "x": float(r["x"])} for r in nrows],
                   [(int(r["source_id"]), int(r["target_id"])) for r in erows])
    print("built a demo submission from ground truth (perfect prediction)")
print("datasets in submission:", sorted(pred))


## 2. Score — with the two parts kept separate

The competition score is

```
adj_edge_jaccard = edge_jaccard * (1 - 0.1 * (N_pred - N_true) / N_true)
final            = adj_edge_jaccard + 0.1 * division_jaccard
```

These two factors have completely different reliability:

- the **multiplier** uses `N_true` from GEFF metadata and your own node count, so
  it is **exact** — no sampling error at all
- `edge_jaccard` is measured on ~2% of your edges, so it is a **sample estimate**

Reporting them merged hides that. We keep them apart.

In [ ]:
rows = []
for m in available:
    nrows, epairs = pred.get(m, ([], []))
    if not nrows:
        print(f"{m}: absent from submission - skipped"); continue
    g  = build_graph(nrows, epairs)
    er = evaluate(g, truth[m]["graph"], scale=truth[m]["scale"], max_distance=7.0)
    rec = node_recall(g, truth[m]["graph"])
    mm = per_sample_metrics(er=er, n_total=truth[m]["n_total"], node_recall=rec)
    D  = er.edge_tp + er.edge_fp + er.edge_fn
    ratio = (len(nrows) - truth[m]["n_total"]) / truth[m]["n_total"]
    rows.append(dict(dataset=m, n_pred=len(nrows), n_true=int(truth[m]["n_total"]),
                     node_ratio=ratio, multiplier=1 - 0.1*ratio,
                     tp=er.edge_tp, fp=er.edge_fp, fn=er.edge_fn, D=D,
                     edge_jaccard=mm["edge_jaccard"], adj=mm["adj_edge_jaccard"],
                     div_tp=er.division_tp, div_fp=er.division_fp, div_fn=er.division_fn))

import pandas as pd
df = pd.DataFrame(rows)
print(df[["dataset","n_pred","n_true","node_ratio","multiplier",
          "tp","fp","fn","D","edge_jaccard","adj"]].to_string(index=False))

Dtot = df.D.sum()
agg_adj = (df.adj * df.D).sum() / Dtot                       # LB weights by TP+FP+FN
dtp, dfp, dfn = df.div_tp.sum(), df.div_fp.sum(), df.div_fn.sum()
div_j = dtp / (dtp + dfp + dfn) if (dtp + dfp + dfn) else float("nan")
print(f"\nweighted adj_edge_jaccard : {agg_adj:.4f}   (weights = D per movie)")
print(f"division jaccard          : {div_j if div_j==div_j else float('nan'):.4f}"
      f"   (pooled {dtp}/{dfp}/{dfn})")
print(f"evaluable edges D         : {Dtot:,}")


## 3. How much of that is real?

`edge_jaccard = TP/(TP+FP+FN)`, so each of the `D` evaluable edges lands in
exactly one of three buckets. We bootstrap over those outcomes: resample `D`
of them with replacement, recompute the weighted aggregate, repeat. The spread
of the result is the sampling uncertainty from the sparse labels.

This does **not** capture bias — only variance. If the annotated cells are
systematically easier than the unannotated ones, the point estimate is off in a
direction the bootstrap cannot see.

In [ ]:
rng = np.random.default_rng(0)
N_BOOT = 2000

def boot_movie(r, rng):
    """Resample D outcomes for one movie; return resampled edge_jaccard."""
    p = np.array([r.tp, r.fp, r.fn], dtype=float)
    if p.sum() == 0: return np.nan
    draw = rng.multinomial(int(r.D), p / p.sum())
    return draw[0] / max(1, draw.sum())

boots = np.empty(N_BOOT)
for b in range(N_BOOT):
    num = den = 0.0
    for r in df.itertuples():
        j = boot_movie(r, rng)
        if j != j: continue
        num += (j * r.multiplier) * r.D      # multiplier is exact, not resampled
        den += r.D
    boots[b] = num / den if den else np.nan

lo, hi = np.nanpercentile(boots, [2.5, 97.5])
half = (hi - lo) / 2
print(f"weighted adj_edge_jaccard : {agg_adj:.4f}")
print(f"95% bootstrap CI          : [{lo:.4f}, {hi:.4f}]")
print(f"resolution (half-width)   : +-{half:.4f}")
print()
print(f"=> Differences smaller than about {2*half:.3f} between two submissions")
print(f"   are NOT distinguishable with these labels.")
print(f"   You have {Dtot:,} evaluable edges; to halve this bound you would need ~{4*Dtot:,}.")


## 4. Where your uncertainty lives

Annotation density is wildly uneven, so some movies contribute almost nothing.
The chart shows each movie's `edge_jaccard` with its own bootstrap interval,
sized by how much weight it carries.

In [ ]:
import matplotlib.pyplot as plt

BLUE, INK, MUTED, GRID = "#2a78d6", "#1a1a19", "#6b6a63", "#e6e5e0"
per_ci = []
for r in df.itertuples():
    bs = np.array([boot_movie(r, rng) for _ in range(N_BOOT)])
    l, h = np.nanpercentile(bs, [2.5, 97.5])
    per_ci.append((r.dataset, r.edge_jaccard, l, h, r.D))

fig, ax = plt.subplots(figsize=(9, 0.9 * len(per_ci) + 1.6))
ys = range(len(per_ci))
for y, (name, j, l, h, D) in zip(ys, per_ci):
    ax.plot([l, h], [y, y], lw=2, color=BLUE, solid_capstyle="round", zorder=2)
    ax.plot([j], [y], "o", ms=9, color=BLUE, zorder=3)
    ax.annotate(f"{j:.3f}  (D={D:,}, {D/Dtot:.0%} of weight)",
                xy=(h, y), xytext=(8, 0), textcoords="offset points",
                va="center", fontsize=9, color=INK)
ax.set_yticks(list(ys)); ax.set_yticklabels([p[0] for p in per_ci], fontsize=9, color=INK)
ax.set_xlabel("edge Jaccard (dot) with 95% bootstrap interval (bar)", fontsize=9, color=MUTED)
ax.set_title("Per-movie estimate and its uncertainty", fontsize=11, color=INK, loc="left", pad=12)
ax.grid(True, axis="x", lw=0.6, color=GRID); ax.set_axisbelow(True)
for s in ("top", "right", "left"): ax.spines[s].set_visible(False)
ax.spines["bottom"].set_color("#d8d7d1"); ax.tick_params(colors=MUTED, labelsize=9)
ax.set_xlim(-0.02, 1.35)
fig.tight_layout(); plt.show()

print("Movies with very few annotated edges have intervals so wide they cannot")
print("discriminate between candidates at all - their apparent score is mostly noise.")


## 5. How to read this

**Use it for:**

- catching invalid or broken graphs before spending a daily submission slot
- detecting large regressions (a change of several points is real)
- the **node-count multiplier**, which is exact — if you are far from `N_true`,
  that is a genuine, measurable problem worth fixing

**Do not use it for:**

- choosing between candidates that differ by less than the resolution bound above
- division tuning — the labelled split contains only a handful of annotated
  divisions, so `division_jaccard` is close to unmeasurable here
- assuming the ranking transfers. The metric ignores edges touching no annotated
  node, so ~98% of your predictions are invisible locally, and the annotated
  subset is plausibly the easier cells. **A local ranking can invert on the
  leaderboard** — that is a real failure mode, not a hypothetical.

**A note on `adj_edge_jaccard > 1`.** The multiplier `1 - 0.1*(N_pred-N_true)/N_true`
exceeds 1 when you predict *fewer* nodes than `N_true` — which counts all cells,
including unannotated ones. That is not a bug; under-prediction is genuinely
rewarded, and it is one of the few levers you can tune with certainty because the
term has no sampling error.

If this saved you a slot, an upvote on the
[dataset](https://www.kaggle.com/datasets/dalloliogm/biohub-official-scorer-patched)
is appreciated. The scoring code is the organisers' work, redistributed
unmodified under BSD-3.